# Fixer — Exchange Rates

Demonstrates **fixer_symbols**, **fixer_latest_rates**, **fixer_historical_rates**, **fixer_convert**, **fixer_timeseries**, and **fixer_fluctuation** tools
using the [Fixer API](https://fixer.io).

Requires a `FIXER_ACCESS_KEY` environment variable set in `.env`.

---
## Setup

In [ ]:
!pip install -q -e "../.."
!pip install -q -r ../requirements.txt

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from demos.shared.llm_setup import get_llm
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from pymcpx.fixer import FixerLatestRatesTool, FixerHistoricalRatesTool, FixerConvertTool, FixerSymbolsTool

llm = get_llm()

tools = [FixerSymbolsTool(), FixerLatestRatesTool(), FixerHistoricalRatesTool(), FixerConvertTool()]

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a currency exchange rate assistant. Use the Fixer tools to "
        "answer questions about exchange rates, conversions, and supported currencies.\n\n"
        "Available tools:\n"
        "- fixer_symbols() \u2014 list supported currencies\n"
        "- fixer_latest_rates(base, symbols) \u2014 latest exchange rates\n"
        "- fixer_historical_rates(date, base, symbols) \u2014 historical rates\n"
        "- fixer_convert(from_, to, amount, date) \u2014 convert between currencies",
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
print("\u2705 Agent ready. Running scenarios...")

### Scenario 1: Latest Exchange Rates

Get the latest exchange rates for major currencies.

In [ ]:
result = executor.invoke({"input": "What are the latest exchange rates for USD, GBP, and JPY against the Euro?"})
print(result["output"])

### Scenario 2: Currency Conversion

Convert a specific amount between currencies.

In [ ]:
result = executor.invoke({"input": "Convert 100 US dollars to Japanese yen using the latest rates."})
print(result["output"])

### Scenario 3: Historical Rates

Check what the exchange rate was on a specific past date.

In [ ]:
result = executor.invoke({"input": "What was the EUR to USD exchange rate on 2024-01-15?"})
print(result["output"])

### Scenario 4: List Supported Currencies

Show all available currency symbols.

In [ ]:
result = executor.invoke({"input": "List all supported currency codes and their full names."})
print(result["output"])